# jikan API

In [ ]:
import requests

def get_search_result_search_results(anime_name, top_n=5):
    url = "https://api.jikan.moe/v4/anime"
    params = {
        "q": anime_name,
        "limit": top_n,
        "order_by": "popularity"
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        return []

    data = response.json().get("data", [])
    results = []

    for anime in data:
        anime_info = {
            "mal_id": anime.get("mal_id"),
            "title": anime.get("title"),
            "name_eng": anime.get("title_english"),
            "name_jap": anime.get("title_japanese"),
            "other_names": anime.get("title_synonyms", []),
            "type": anime.get("type"),
            "genres": [genre["name"] for genre in anime.get("genres", [])],
            "explicit_genres": [genre["name"] for genre in anime.get("explicit_genres", [])],
            "themes": [genre["name"] for genre in anime.get("themes", [])],
            "demographics": [genre["name"] for genre in anime.get("demographics", [])],
            "studio": [studio["name"] for studio in anime.get("studios", [])],
            "age_rating": anime.get("rating"),
            "description": anime.get("synopsis"),
            "original_source": anime.get("source"),
            "cover_image": anime.get("images", {}).get("jpg", {}).get("large_image_url"),
            "score": anime.get("score"),
            "scored_by": anime.get("scored_by"),
            "episodes": anime.get("episodes"),
            "season": f"{anime.get('season', 'Unknown')} {anime.get('year', 'Unknown')}",
            "status": anime.get("status"),
            "duration": anime.get("duration")
        }
        results.append(anime_info)

    return results

# 🔍 Example usage:
top_results = get_search_result_search_results("My hero", top_n=5)
for i, anime in enumerate(top_results, 1):
    print(f"\nResult {i}:")
    for key, value in anime.items():
        print(f"{key}: {value}")

In [ ]:
import requests

def get_anime_id(anime_name):
    url = "https://api.jikan.moe/v4/anime"
    params = {"q": anime_name, "limit": 1}
    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Search failed: {response.status_code}")
        return None
    data = response.json().get("data", [])
    return data[0]["mal_id"] if data else None

def get_anime_relations(mal_id):
    url = f"https://api.jikan.moe/v4/anime/{mal_id}/relations"
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Relations fetch failed: {response.status_code}")
        return []
    data = response.json().get("data", [])
    
    related_list = []
    for relation in data:
        relation_type = relation.get("relation")
        for entry in relation.get("entry", []):
            related_list.append({
                "relation_type": relation_type,
                "mal_id": entry.get("mal_id"),
                "title": entry.get("title"),
                "type": entry.get("type"),
                "url": entry.get("url")
            })
    return related_list

# 🔍 Example usage:
anime_name = "My Hero Academia: Hero Notebook"
main_id = get_anime_id(anime_name)
if main_id:
    related_anime = get_anime_relations(main_id)
    print(f"\nMulti-Season Timeline or Related Titles for '{anime_name}':")
    for rel in related_anime:
        # print(rel)
        print(f"[{rel['relation_type']}] {rel['mal_id']} ({rel['type']}) - {rel['url']}")


In [ ]:
import requests

def get_all_anime_genres():
    url = "https://api.jikan.moe/v4/genres/anime"
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to fetch genres: {response.status_code}")
        return []

    data = response.json().get("data", [])
    genres = [{"id": genre["mal_id"], "name": genre["name"], "url": genre["url"]} for genre in data]
    return genres

# 🔍 Example usage:
anime_genres = get_all_anime_genres()
print("🎭 All available anime genres:")
for genre in anime_genres:
    print(f"- {genre['name']} (ID: {genre['id']})")


In [ ]:
import requests
import time

BASE_URL = "https://api.jikan.moe/v4"

def fetch_anime_by_id(mal_id, delay=0.5, retries=5):
    url = f"{BASE_URL}/anime/{mal_id}"
    for i in range(retries):
        try:
            response = requests.get(url)
            if response.status_code == 404:
                return None
            response.raise_for_status()
            return response.json().get("data", {})
        except requests.exceptions.HTTPError as e:
            if response.status_code == 429:
                time.sleep(2 ** i)  # exponential backoff
            else:
                return None
        time.sleep(delay)
    return None

def fetch_relations(mal_id):
    try:
        response = requests.get(f"{BASE_URL}/anime/{mal_id}/relations")
        response.raise_for_status()
        return response.json().get("data", [])
    except Exception:
        return []

def walk_season_chain(mal_id, visited=None):
    if visited is None:
        visited = set()
    if mal_id in visited:
        return []

    visited.add(mal_id)
    anime_data = fetch_anime_by_id(mal_id)
    if not anime_data:
        return []

    chain = [anime_data]

    relations = fetch_relations(mal_id)
    for rel in relations:
        if rel["relation"].lower() in {"sequel", "prequel"}:
            for entry in rel.get("entry", []):
                next_id = entry["mal_id"]
                if next_id not in visited:
                    chain += walk_season_chain(next_id, visited)
    return chain

def get_main_series_chain(start_title):
    # Step 1: Search for the anime
    search = requests.get(f"{BASE_URL}/anime", params={"q": start_title, "limit": 1})
    search.raise_for_status()
    results = search.json().get("data", [])
    if not results:
        return {"error": "Anime not found."}

    start_anime = results[0]
    start_id = start_anime["mal_id"]

    # Step 2: Walk the full sequel/prequel chain
    full_chain = walk_season_chain(start_id)

    # Step 3: Deduplicate and sort by air date (or fallback to mal_id)
    seen = set()
    unique_chain = []
    for anime in full_chain:
        if anime["mal_id"] not in seen:
            seen.add(anime["mal_id"])
            unique_chain.append(anime)

    def sort_key(a):
        date = a.get("aired", {}).get("from")
        return date or "9999-99-99"

    sorted_chain = sorted(unique_chain, key=sort_key)

    # Step 4: Format result
    result = {
        "franchise_title": sorted_chain[0].get("title_english") or sorted_chain[0].get("title"),
        "main_series": [
            {
                "title": anime.get("title_english") or anime.get("title"),
                "mal_id": anime["mal_id"],
                "aired": anime.get("aired", {}).get("from"),
                "episodes": anime.get("episodes"),
                "type": anime.get("type"),
            }
            for anime in sorted_chain
        ]
    }

    return result


data = get_main_series_chain("My Hero Academia")

print("FRANCHISE TITLE:", data["franchise_title"])
print("\nMAIN SERIES:")
for anime in data["main_series"]:
    print(f"- {anime['title']} ({anime['aired']})")


In [ ]:
data

## without vibe-coding

In [ ]:
import requests
from retrying import retry

BASE_URL = "https://api.jikan.moe/v4"

def extract_information(anime: dict) -> dict:
    anime_info = {
        "mal_id": anime.get("mal_id"),
        "mal_url": anime.get("url"),
        "name_eng": anime.get("title_english"),
        "name_jap": anime.get("title_japanese"),
        "other_names": anime.get("title_synonyms", []),
        "type": anime.get("type"),
        "genres": [genre["name"] for genre in anime.get("genres", [])],
        "explicit_genres": [genre["name"] for genre in anime.get("explicit_genres", [])],
        "themes": [genre["name"] for genre in anime.get("themes", [])],
        "demographics": [genre["name"] for genre in anime.get("demographics", [])],
        "studio": [studio["name"] for studio in anime.get("studios", [])],
        "age_rating": anime.get("rating"),
        "description": anime.get("synopsis"),
        "original_source": anime.get("source"),
        "cover_image": anime.get("images", {}).get("jpg", {}).get("large_image_url"),
        "score": anime.get("score"),
        "scored_by": anime.get("scored_by"),
        "episodes": anime.get("episodes"),
        "season": f"{anime.get('season', 'Unknown')} {anime.get('year', 'Unknown')}",
        "aired": anime.get("aired", {}).get("from"),
        "status": anime.get("status"), # e.g. "Finished Airing"
        "duration": anime.get("duration")
    }
    return anime_info

@retry(
    stop_max_attempt_number=3,  # Retry up to 3 times
    wait_fixed=1000            # Wait 1s between retries
)
def fetch_relations(mal_id):
    response = requests.get(f"{BASE_URL}/anime/{mal_id}/relations")
    response.raise_for_status()
    return response.json().get("data", [])

@retry(
    stop_max_attempt_number=3,  # Retry up to 3 times
    wait_fixed=1000            # Wait 1s between retries
)
def search_by_malid(mal_id):
    response = requests.get(f"{BASE_URL}/anime/{mal_id}")
    response.raise_for_status()
    return response.json().get("data", {})

def get_all_anime_media(media_list: list[dict]):
    mal_ids = []
    for media in media_list:
        if media["type"] == 'anime':
            mal_ids.append(media["mal_id"])
    return mal_ids

def get_relation_type(is_main_season: bool, relation_type):
    if is_main_season:
        return "main"
    else:
        if relation_type is not None:
            return relation_type
        else:
            return "other"

def search_title(title: str):
    # Step 1: Search for the anime
    search = requests.get(f"{BASE_URL}/anime", params={"q": title, "limit": 3})
    search.raise_for_status()
    results = search.json().get("data", [])
    if not results:
        return {"error": "Anime not found."}
    
    all_info = {}
    relations = []
    for anime in results:
        anime_info = extract_information(anime)
        print(f"### {anime_info['name_eng']} ###")
        mal_id = anime_info["mal_id"]
        related_anime_graph = {}
        left_mal_ids = [mal_id]
        relation_type = [None]
        while len(left_mal_ids) > 0:
            current_mal_id = left_mal_ids[0]
            if not current_mal_id in all_info:
                if mal_id != current_mal_id: # Skip redundant fetching for original three fetched animes
                    anime_info = extract_information(search_by_malid(current_mal_id))
                if anime_info["type"].lower() != "music": # Filter out music videos etc.
                    all_info[anime_info["mal_id"]] = anime_info

                    all_related_media = fetch_relations(current_mal_id)
                    relation_types = [related_media["relation"] for related_media in all_related_media]
                    is_main_season = (relation_type[0] is None) and (anime_info["type"].lower() == "tv") and ("full_story" not in relation_types) and ("parent_story" not in relation_types)
                    
                    related_anime_graph[anime_info["mal_id"]] = {"mal_id": current_mal_id, "title": anime_info["name_eng"], "aired": anime_info["aired"], "type": anime_info["type"], "relation_type": get_relation_type(is_main_season, relation_type[0])}

                    for related_media in all_related_media:
                        if related_media["relation"].lower() in ['summary', 'character']:
                            special_relation_mal_ids = get_all_anime_media(related_media["entry"])
                            left_mal_ids += special_relation_mal_ids
                            relation_type += [related_media["relation"].lower()] * len(special_relation_mal_ids)
                        elif related_media["relation"].lower() != 'adaptation':
                            other_mal_ids = get_all_anime_media(related_media["entry"])
                            left_mal_ids += other_mal_ids
                            relation_type += [None] * len(other_mal_ids)

            left_mal_ids.remove(current_mal_id)
            relation_type.remove(relation_type[0])

        if related_anime_graph:
            # Sort by aired date (None dates go last)
            sorted_items = sorted(
                related_anime_graph.items(),
                key=lambda item: (item[1]["aired"] is None, item[1]["aired"])
            )
            sorted_graph = dict(sorted_items)
            relations.append(sorted_graph)
            
    return relations, all_info

search_title("My Hero")

In [ ]:
import sys
import httpx
from collections import deque
from typing import Optional
from tenacity import retry, stop_after_attempt, wait_fixed, before_sleep_log
import logging

BASE_URL = "https://api.jikan.moe/v4"

logging_config = {
    "level": logging.DEBUG,  # Set to DEBUG during development, INFO/WARNING in prod
    "format": "%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    "handlers": [
        logging.StreamHandler(sys.stdout)
    ],
}
logging.basicConfig(**logging_config)

logger = logging.getLogger(__name__)


class AnimeNotFoundError(Exception):
    def __init__(self, title: str):
        super().__init__(f"Anime titled '{title}' not found.")


class JikanScraper:
    def __init__(self, base_url: str = BASE_URL):
        self.base_url = base_url
        self.client: Optional[httpx.AsyncClient] = None

    async def __aenter__(self):
        self.client = httpx.AsyncClient()
        return self

    async def __aexit__(self, exc_type, exc, tb):
        await self.client.aclose()

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_fixed(1),
        before_sleep=before_sleep_log(logger, logging.DEBUG)
    )
    async def _get(self, url: str, params: Optional[dict] = None) -> dict:
        logger.debug(f"Fetching URL: {url} with params: {params}")
        response = await self.client.get(url, params=params)
        response.raise_for_status()
        return response.json()

    def extract_information(self, anime: dict) -> dict:
        return {
            "mal_id": anime.get("mal_id"),
            "mal_url": anime.get("url"),
            "title": anime.get("title"),
            "name_eng": anime.get("title_english"),
            "name_jap": anime.get("title_japanese"),
            "other_names": anime.get("title_synonyms", []),
            "type": anime.get("type"),
            "genres": [genre["name"] for genre in anime.get("genres", [])],
            "explicit_genres": [genre["name"] for genre in anime.get("explicit_genres", [])],
            "themes": [genre["name"] for genre in anime.get("themes", [])],
            "demographics": [genre["name"] for genre in anime.get("demographics", [])],
            "studio": [studio["name"] for studio in anime.get("studios", [])],
            "age_rating": anime.get("rating"),
            "description": anime.get("synopsis"),
            "original_source": anime.get("source"),
            "cover_image": anime.get("images", {}).get("jpg", {}).get("large_image_url"),
            "score": anime.get("score"),
            "scored_by": anime.get("scored_by"),
            "episodes": anime.get("episodes"),
            "season": f"{anime.get('season', 'Unknown')} {anime.get('year', 'Unknown')}",
            "aired": anime.get("aired", {}).get("from"),
            "status": anime.get("status"),
            "duration": anime.get("duration"),
        }

    async def fetch_relations(self, mal_id: int) -> list[dict]:
        data = await self._get(f"{self.base_url}/anime/{mal_id}/relations")
        return data.get("data", [])

    async def search_by_malid(self, mal_id: int) -> dict:
        data = await self._get(f"{self.base_url}/anime/{mal_id}")
        return data.get("data", {})

    def get_all_anime_media(self, media_list: list[dict]) -> list[int]:
        return [media["mal_id"] for media in media_list if media.get("type") == "anime"]

    def get_relation_type(self, is_main_season: bool, relation_type: Optional[str]) -> str:
        return "main" if is_main_season else (relation_type or "other")

    async def search_title(self, title: str) -> tuple[list[dict], dict[int, dict]]:
        search = await self._get(f"{self.base_url}/anime", params={"q": title, "limit": 3})
        results = search.get("data", [])

        if not results:
            raise AnimeNotFoundError(title)

        all_info: dict[int, dict] = {}
        visited_ids: set[int] = set()
        relations: list[dict] = []

        for anime in results:
            anime_info = self.extract_information(anime)
            logger.info(f"Searching relations with: {anime_info['title']}")
            mal_id = anime_info["mal_id"]
            related_anime_graph: dict[int, dict] = {}
            left_mal_ids = deque([mal_id])
            relation_types = deque([None])

            while left_mal_ids:
                current_mal_id = left_mal_ids.popleft()
                current_relation = relation_types.popleft()

                if current_mal_id in visited_ids:  # skip if visited
                    continue
                
                logger.info(f"Media left: {len(set(left_mal_ids) - visited_ids)}")
                # Mark as visited BEFORE doing anything
                visited_ids.add(current_mal_id)

                if mal_id != current_mal_id:
                    anime_info = self.extract_information(await self.search_by_malid(current_mal_id))

                if anime_info["type"]:
                    if anime_info["type"].lower() == "music":
                        logger.debug(f"Skipping anime music: {anime_info['title']}")
                        continue

                    all_info[anime_info["mal_id"]] = anime_info

                    all_related_media = await self.fetch_relations(current_mal_id)
                    relation_types_list = [media["relation"] for media in all_related_media]

                    is_main_season = (
                        current_relation is None
                        and anime_info["type"].lower() == "tv"
                        and "full_story" not in relation_types_list
                        and "parent_story" not in relation_types_list
                    )

                    related_anime_graph[anime_info["mal_id"]] = {
                        "mal_id": current_mal_id,
                        "title": anime_info["title"],
                        "aired": anime_info["aired"],
                        "type": anime_info["type"],
                        "relation_type": self.get_relation_type(is_main_season, current_relation),
                    }

                    for related_media in all_related_media:
                        rel = related_media["relation"].lower()
                        if rel == 'character': # Skip because not related enough
                            continue

                        if rel != "adaptation":
                            mal_ids = self.get_all_anime_media(related_media["entry"])
                            left_mal_ids.extend(mal_ids)
                            relation_types.extend([rel] * len(mal_ids))
                            if rel in ["summary"]:
                                relation_types.extend([rel] * len(mal_ids))
                            else:
                                relation_types.extend([None] * len(mal_ids))

                else:
                    logger.warning(f"Anime without type:\n\n{anime_info}")

            if related_anime_graph:
                sorted_graph = dict(
                    sorted(
                        related_anime_graph.items(),
                        key=lambda item: (item[1]["aired"] is None, item[1]["aired"])
                    )
                )
                relations.append(sorted_graph)

        return relations, all_info


async with JikanScraper() as scraper:
    results = await scraper.search_title("MyHero") # "My Hero" -> ~51s, "Naruto" -> 10+ min, "Eminence in Shadow" -> ~46s
results

In [ ]:
def get_first_main_relation(media_dict: dict) -> dict | None:
    for mal_id, media in media_dict.items():
        if media.get("relation_type") == "main":
            return mal_id
    return None  # If no main relation found

get_first_main_relation(results[0][0])